## Phase 5 — SQL Business Analysis

### Objective

Use SQL to analyze the cleaned loan dataset and answer business questions related to loan default risk.

The analysis will focus on borrower characteristics, loan characteristics, default rates, and risk patterns.

### Business Goal

Identify patterns in loan defaults that can help understand which borrower and loan segments may represent higher credit risk.

### Approach

The cleaned dataset from Phase 4 will be loaded into a SQL environment.

Business questions will be answered using SQL queries, and the results will be interpreted from a credit-risk perspective.

### 5.1 SQL Environment Setup

SQLite will be used as the SQL engine for this phase.

The cleaned loan dataset will be loaded into a SQLite database table so that business questions can be answered using SQL.

SQLite is used because it is lightweight, requires no separate database server, and integrates directly with Python through the built-in `sqlite3` module.

In [2]:
import sqlite3

conn = sqlite3.connect('../data/processed/credit_risk.db')

print("SQLite database connection established.")

SQLite database connection established.


### 5.2 Load Cleaned Dataset into SQLite

The cleaned dataset from Phase 4 will be loaded into a SQLite table named `loans`.

The CSV remains the source file, while SQLite provides the environment for performing SQL-based business analysis.

In [3]:
import pandas as pd

cleaned_path = '../data/processed/lc_loans_granting_model_dataset_cleaned.csv'

df_clean = pd.read_csv(cleaned_path)

df_clean.to_sql(
    'loans',
    conn,
    if_exists='replace',
    index=False
)

print("DataFrame written to SQLite database as 'loans' table.")

DataFrame written to SQLite database as 'loans' table.


In [4]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,loans


### 5.3 Business Question 1 — Overall Default Rate

### Business Question

What percentage of loans in the portfolio resulted in default?

### Business Purpose

The overall default rate provides a baseline measure of credit risk in the portfolio.

This baseline can later be used to compare default rates across different borrower and loan segments.

### SQL Approach

- Count the total number of loans.
- Count the number of loans where `Default = 1`.
- Calculate the default rate as:

`Defaulted Loans / Total Loans × 100`

In [6]:
result = pd.read_sql_query(
    """
    SELECT
    COUNT(*) AS total_loans
    FROM loans;
    """,
    conn
)

result

,total_loans
0,1347681


In [15]:
default_loans = pd.read_sql_query(
    """
    SELECT
    'Default',
    COUNT(*) AS default_loans
    FROM loans
    WHERE "Default" == 1;
    """,
    conn
)

default_loans

,'Default',default_loans
0,Default,269249


In [16]:
default_values = pd.read_sql_query(
    """
    SELECT
        "Default",
        COUNT(*) AS loan_count
    FROM loans
    GROUP BY "Default"
    ORDER BY "Default";
    """,
    conn
)

default_values

,Default,loan_count
0,0,1078432
1,1,269249


In [19]:
overall_default_rate = pd.read_sql_query(
    """
    SELECT
    COUNT(*) AS total_loans,
    SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) AS default_loans,
    ROUND(
        SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) * 100.0
        / COUNT(*),2
        ) AS default_rate_pct
    FROM loans;
    """,
    conn
)

overall_default_rate

,total_loans,default_loans,default_rate_pct
0,1347681,269249,19.98


### Interpretation

The overall default rate is **19.98%**, meaning approximately 20 out of every 100 loans in the dataset resulted in default.

This value establishes the baseline default rate for the portfolio.

Future segment-level analysis will compare individual borrower and loan groups against this baseline to identify segments with relatively higher or lower default risk.

### 5.4 Business Question 2 — Default Rate by Loan Purpose

#### Business Question

Which loan purposes have higher or lower default rates?

#### Business Purpose

Different loan purposes may have different levels of credit risk.

Calculating the default rate for each loan purpose allows us to compare risk across loan categories while accounting for differences in the number of loans.

#### SQL Approach

For each `purpose`:

- Count total loans.
- Count defaulted loans.
- Calculate the default rate as:

`Defaulted Loans / Total Loans × 100`

The results will be compared with the overall portfolio default rate of 19.98%.

In [22]:
default_by_purpose = pd.read_sql_query(
    """
    SELECT
        purpose,
        COUNT(*) AS total_loans,
        SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) AS default_loans,
        ROUND(
            SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),2
            ) AS default_rate_pct
        FROM loans
        GROUP BY purpose
        ORDER BY default_rate_pct DESC;
        """,
        conn
)


default_by_purpose

,purpose,total_loans,default_loans,default_rate_pct
0,small_business,15575,4650,29.86
1,renewable_energy,936,222,23.72
2,moving,9520,2229,23.41
3,house,7294,1598,21.91
4,medical,15606,3409,21.84
5,debt_consolidation,781206,165258,21.15
6,other,78263,16501,21.08
7,educational,423,88,20.80
8,vacation,9083,1743,19.19
9,major_purchase,29543,5496,18.60


### Business Question 2 — Default Rate by Loan Purpose

#### Finding

Default rates vary considerably across loan purposes.

The overall portfolio default rate is 19.98%.

The highest default rate is observed for `small_business` loans at 29.86%, which is 9.88 percentage points above the overall portfolio rate.

`renewable_energy`, `moving`, `house`, and `medical` also have default rates above the portfolio average.

`wedding` loans have the lowest observed default rate at 12.43%.

#### Business Interpretation

Loan purpose appears to be associated with different levels of default risk.

However, loan purpose alone should not be interpreted as a causal driver of default. Differences in borrower characteristics, credit scores, income, loan amounts, and other factors may contribute to these patterns.

From a portfolio perspective, `debt_consolidation` is particularly important because it represents the largest loan-purpose segment with 781,206 loans and has a 21.15% default rate, slightly above the overall portfolio default rate.

In [24]:
default_by_FICO = pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN fico_n < 650 THEN 'Poor'
            WHEN fico_n BETWEEN 650 AND 699 THEN 'Fair'
            WHEN fico_n BETWEEN 700 AND 749 THEN 'Good'
            ELSE 'Excelent'
        END AS fico_category,
        COUNT(*) AS total_loans,
        SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) AS defaulted_loans,
        ROUND(
            SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
            ) AS default_rate_pct
        FROM loans
        GROUP BY fico_category
        ORDER BY default_rate_pct DESC;
        """,
        conn
)


default_by_FICO

,fico_category,total_loans,defaulted_loans,default_rate_pct
0,Poor,231,74,32.03
1,Fair,822284,194083,23.60
2,Good,418569,65624,15.68
3,Excelent,106597,9468,8.88


### Business Question 3 — Default Rate by FICO Category

#### Finding

Default rates show a clear relationship with FICO category.

- Poor (<650): 32.03%
- Fair (650–699): 23.60%
- Good (700–749): 15.68%
- Excellent (750+): 8.88%

The overall portfolio default rate is 19.98%.

Borrowers in the Poor and Fair categories have default rates above the portfolio average, while Good and Excellent borrowers have default rates below the portfolio average.

#### Business Interpretation

FICO score appears to be strongly associated with observed default risk in the portfolio.

As FICO category improves, the observed default rate decreases substantially.

The Poor category has only 231 loans, so its default rate should be interpreted with caution because of its relatively small sample size.

From a credit-risk perspective, FICO score can be considered an important risk-segmentation variable and should be investigated further during statistical analysis and machine-learning phases.

In [31]:
default_by_dti = pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN dti_n < 10 THEN '<10'
            WHEN dti_n BETWEEN 10 AND 19.99 THEN '10-20'
            WHEN dti_n BETWEEN 20 AND 29.99 THEN '20-30'
            WHEN dti_n BETWEEN 30 AND 39.99 THEN '30-40'
            ELSE '40+'
        END AS dti_category,

        COUNT(*) AS total_loans,

        SUM(
            CASE
                WHEN "Default" = 1 THEN 1
                ELSE 0
            END
        ) AS defaulted_loans,

        ROUND(
            SUM(
                CASE
                    WHEN "Default" = 1 THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*),
            2
        ) AS default_rate_pct

    FROM loans

    WHERE dti_n IS NOT NULL

    GROUP BY dti_category

    ORDER BY default_rate_pct DESC;
    """,
    conn
)

default_by_dti

,dti_category,total_loans,defaulted_loans,default_rate_pct
0,40+,6770,2079,30.71
1,30-40,122297,35610,29.12
2,20-30,410172,94646,23.07
3,10-20,563230,100517,17.85
4,<10,245174,36388,14.84


### Business Question 4 — Default Rate by DTI Category

Debt-to-income (DTI) ratio is an important credit-risk indicator because it reflects the borrower's debt burden relative to income.

This analysis groups borrowers into DTI categories and compares the number of loans, defaulted loans, and default rate across the groups.

### Business Question

Does a higher debt-to-income ratio correspond to a higher loan default rate?

### Key Finding

The results show a clear increasing pattern in default rates as DTI increases.

- Borrowers with DTI below 10 have a default rate of **14.84%**.
- Borrowers with DTI between 10 and 20 have a default rate of **17.85%**.
- Borrowers with DTI between 20 and 30 have a default rate of **23.07%**.
- Borrowers with DTI between 30 and 40 have a default rate of **29.12%**.
- Borrowers with DTI of 40 or higher have the highest default rate at **30.71%**.

This indicates that higher DTI levels are associated with higher observed loan default rates in the dataset.

The DTI 40+ category has a smaller number of loans than the other major categories, so both default rate and segment size should be considered when evaluating credit risk.

In [25]:
default_by_loan_amount = pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN loan_amnt < 5000 THEN '<5k'
            WHEN loan_amnt BETWEEN 5000 AND 10000 THEN '5k-10k'
            WHEN loan_amnt BETWEEN 10000 AND 20000 THEN '10k-20k'
            WHEN loan_amnt BETWEEN 20000 AND 30000 THEN '20k-30k'
            ELSE '30k+'
        END AS loan_amnt_band,
        COUNT(*) AS total_loan,
        SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) AS defaulted_loans,
        ROUND(
            SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),2
            ) AS default_rate_pct
        FROM loans
        GROUP BY loan_amnt_band
        ORDER BY default_rate_pct DESC;
        """,
        conn
)

default_by_loan_amount

,loan_amnt_band,total_loan,defaulted_loans,default_rate_pct
0,30k+,81325,19911,24.48
1,20k-30k,210603,48329,22.95
2,10k-20k,498432,107545,21.58
3,5k-10k,423474,72687,17.16
4,<5k,133847,20777,15.52


### Business Question 5 — Default Rate by Loan Amount

To understand whether larger loans are associated with higher default risk, loan amounts are grouped into five bands:

- `<5k`
- `5k–10k`
- `10k–20k`
- `20k–30k`
- `30k+`

The default rate is calculated for each loan amount band.

### Business Interpretation

The results show a clear increase in default rate as loan amount increases.

- `<5k`: 15.52%
- `5k–10k`: 17.16%
- `10k–20k`: 21.58%
- `20k–30k`: 22.95%
- `30k+`: 24.48%

The `30k+` loan segment has the highest observed default rate at **24.48%**, while loans below `5k` have the lowest default rate at **15.52%**.

This indicates that larger loan amounts are associated with higher observed default rates in this dataset.

However, this is an observed association and does not establish that loan amount alone causes higher default risk. Other borrower characteristics, such as FICO score, income, and debt-to-income ratio, may also contribute to the relationship.

In [26]:
default_by_emp_length = pd.read_sql_query(
    """
    SELECT
        emp_length,
        COUNT(*) AS total_loans,
        SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) AS defaulted_loans,
        ROUND(
            SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) *100.0
            /COUNT(*),2
        ) AS default_rate_pct
    FROM loans
    GROUP BY emp_length
    ORDER BY default_rate_pct DESC;
    """,
    conn
)

default_by_emp_length

,emp_length,total_loans,defaulted_loans,default_rate_pct
0,NI,78188,21077,26.96
1,1 year,88841,18289,20.59
2,< 1 year,108517,22289,20.54
3,3 years,107862,21558,19.99
4,8 years,60808,12132,19.95
5,9 years,51019,10158,19.91
6,2 years,122091,24197,19.82
7,4 years,80761,15956,19.76
8,5 years,84326,16545,19.62
9,7 years,59724,11652,19.51


### Business Question 6 — Inte—rpretation  Employment Length

The analysis shows differences in observed default rates across employment-length categories.

The `NI` category has the highest default rate at **26.96%**, while borrowers with `10+ years` of employment have the lowest default rate at **18.80%**.

Among borrowers with a stated employment length, the default rate is generally higher for borrowers with shorter employment histories and tends to be lower among borrowers with longer employment histories.

This suggests that employment stability may be associated with observed default risk in the dataset.

However, `NI` is a separate category whose exact meaning was not established during Data Quality analysis, so its high default rate should not automatically be interpreted as unemployment or missing employment information.

Employment length should therefore be considered alongside other borrower characteristics, such as FICO score, income, and debt-to-income ratio, when assessing credit risk.

In [27]:
default_by_home_ownership = pd.read_sql_query(
    """
    SELECT
        home_ownership_n,
        COUNT(*) AS total_loans,
        SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) AS defaulted_loans,
        ROUND(
            SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS default_rate_pct
    FROM loans
    GROUP BY home_ownership_n
    ORDER BY default_rate_pct DESC;
    """,
    conn
)

default_by_home_ownership

,home_ownership_n,total_loans,defaulted_loans,default_rate_pct
0,RENT,535585,124408,23.23
1,OWN,144985,29904,20.63
2,OTHER,517,102,19.73
3,MORTGAGE,666594,114835,17.23


### Business Question 7 — Business Interpretation — Home Ownership

The analysis shows differences in observed default rates across home ownership categories.

Borrowers who **rent** have the highest default rate at **23.23%**, while borrowers with a **mortgage** have the lowest default rate at **17.23%**.

Borrowers who own their homes have an observed default rate of **20.63%**.

The `OTHER` category has a default rate of **19.73%**, but it contains only 517 loans. Therefore, its rate should be interpreted cautiously because of the small sample size.

Overall, the results indicate that housing status is associated with different observed levels of default risk in this dataset. However, this relationship does not imply that home ownership itself causes default risk. Other borrower and loan characteristics may also explain these differences.

In [32]:
default_by_state = pd.read_sql_query(
    """
    SELECT
        addr_state,
        COUNT(*) AS total_loans,

        SUM(
            CASE
                WHEN "Default" = 1 THEN 1
                ELSE 0
            END
        ) AS defaulted_loans,

        ROUND(
            SUM(
                CASE
                    WHEN "Default" = 1 THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*),
            2
        ) AS default_rate_pct

    FROM loans

    GROUP BY addr_state

    ORDER BY default_rate_pct DESC;
    """,
    conn
)

default_by_state

,addr_state,total_loans,defaulted_loans,default_rate_pct
0,MS,6593,1722,26.12
1,NE,3591,905,25.20
2,AR,10058,2425,24.11
3,AL,16640,3933,23.64
4,OK,12291,2884,23.46
5,LA,15520,3596,23.17
6,NY,110075,24267,22.05
7,NV,20282,4457,21.98
8,FL,95815,20599,21.50
9,IN,21720,4656,21.44


### Business Question 8 — Default Rate by State

### Business Question

Which states have the highest and lowest loan default rates?

### Objective

Analyze default rates across borrower states to identify geographic patterns in loan default behavior.

### Method

For each state, calculate:

- Total number of loans
- Number of defaulted loans
- Default rate (%)

The results are ordered from highest to lowest default rate.

### SQL Logic

The analysis groups loans by `addr_state` and calculates the default rate using the number of defaulted loans divided by total loans.

### Business Interpretation

The results show variation in default rates across states.

States with higher observed default rates may represent segments that require further investigation. However, state-level default rates should be interpreted together with other borrower and loan characteristics because differences may be influenced by factors such as credit score, debt-to-income ratio, loan amount, and borrower profile.

### Key Insight

State-level analysis can help identify geographic patterns in credit risk, but geography alone should not be treated as a sufficient basis for making lending decisions.

The state-level results can be combined with other risk characteristics in later phases to develop a more comprehensive view of borrower risk.

In [28]:
default_by_year = pd.read_sql_query(
    """
    SELECT
        SUBSTR(issue_d, 1, 4) AS issue_year,
         COUNT(*) AS total_loans,
        SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) AS defaulted_loans,
        ROUND(
            SUM(CASE WHEN "Default" = 1 THEN 1 ELSE 0 END) * 100.0
            / COUNT(*),
            2
        ) AS default_rate_pct
    FROM loans
    GROUP BY issue_year
    ORDER BY issue_year;
    """,
    conn
)

default_by_year


,issue_year,total_loans,defaulted_loans,default_rate_pct
0,2007,599,158,26.38
1,2008,2393,496,20.73
2,2009,5281,723,13.69
3,2010,12537,1757,14.01
4,2011,21721,3297,15.18
5,2012,53367,8644,16.20
6,2013,134804,21024,15.60
7,2014,223102,41161,18.45
8,2015,375543,75802,20.18
9,2016,293057,68229,23.28


### Business Question 9 — Interpretation — Default Rate Over Time

The observed default rate varies substantially across loan-issue years.

The highest default rate was observed in **2007 at 26.38%**, while the lowest was observed in **2009 at 13.69%**.

From 2009 onward, the default rate generally increased over several periods, reaching **23.28% in 2016** and **23.13% in 2017**.

The default rate then decreased to **15.75% in 2018**.

The relatively small number of loans in the earliest years, particularly 2007 and 2008, should be considered when interpreting their default rates.

Overall, the results indicate that observed default rates changed over time rather than remaining constant across loan-issue periods. This suggests that time-related factors may be relevant to credit-risk analysis and should be considered in later statistical and machine-learning phases.

This analysis identifies an association over time and does not establish that the issue year itself causes changes in default risk.

### Business Question 10 — Loan Amount by Default Status

### Business Question

Do borrowers who default have a different average loan amount compared with borrowers who do not default?

### Objective

Compare the average loan amount between defaulted and non-defaulted loans to determine whether loan size is associated with default behavior.

### Method

Group loans by the `Default` variable and calculate:

- Number of loans
- Average loan amount
- Minimum loan amount
- Maximum loan amount

### Business Interpretation

Comparing loan amounts between defaulted and non-defaulted borrowers can help identify whether larger or smaller loans are associated with different default behavior.

However, average loan amount alone does not establish that loan size causes default. The relationship should be evaluated together with other borrower characteristics such as FICO score, DTI, income, and employment history.

### Key Insight

Loan amount can be considered as one component of credit-risk assessment, but it should not be evaluated independently when making lending decisions.

In [33]:
avg_loan_by_default = pd.read_sql_query(
    """
    SELECT
        "Default",
        COUNT(*) AS total_loans,
        ROUND(AVG(loan_amnt), 2) AS average_loan_amount,
        MIN(loan_amnt) AS minimum_loan_amount,
        MAX(loan_amnt) AS maximum_loan_amount
    FROM loans
    GROUP BY "Default"
    ORDER BY "Default";
    """,
    conn
)

avg_loan_by_default

,Default,total_loans,average_loan_amount,minimum_loan_amount,maximum_loan_amount
0,0,1078432,14123.91,500,40000
1,1,269249,15547.02,500,40000


### Business Question 11 — Higher-Risk Borrower Segments

The previous analyses examined individual factors associated with loan default.

This analysis combines **FICO score category** and **loan amount band** to identify borrower segments with comparatively higher observed default rates.

### Business Question

Which combinations of borrower credit quality and loan amount show higher observed default rates?

Only segments with at least 1,000 loans will be considered to avoid drawing conclusions from very small groups.

In [30]:
higher_risk_segment = pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN fico_n < 650 THEN 'Poor'
            WHEN fico_n BETWEEN 650 AND 699 THEN 'Fair'
            WHEN fico_n BETWEEN 700 AND 749 THEN 'Good'
            ELSE 'Excellent'
        END AS fico_category,

        CASE
            WHEN loan_amnt < 5000 THEN '<5k'
            WHEN loan_amnt BETWEEN 5000 AND 9999 THEN '5k-10k'
            WHEN loan_amnt BETWEEN 10000 AND 19999 THEN '10k-20k'
            WHEN loan_amnt BETWEEN 20000 AND 29999 THEN '20k-30k'
            ELSE '30k+'
        END AS loan_amount_band,

        COUNT(*) AS total_loans,

        SUM(
            CASE
                WHEN "Default" =1 THEN 1 
                ELSE 0 
            END
        ) AS default_loans,
        
         ROUND(
            SUM(
                CASE
                    WHEN "Default" = 1 THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*),
            2
        ) AS default_rate_pct

    FROM loans

    GROUP BY fico_category, loan_amount_band

    HAVING COUNT(*) >= 1000

    ORDER BY default_rate_pct DESC;
""",
conn
)

higher_risk_segment

,fico_category,loan_amount_band,total_loans,default_loans,default_rate_pct
0,Fair,30k+,55656,16532,29.70
1,Fair,20k-30k,136921,38103,27.83
2,Fair,10k-20k,330115,82585,25.02
3,Good,30k+,43617,8496,19.48
4,Fair,5k-10k,213072,41411,19.44
5,Fair,<5k,86520,15452,17.86
6,Good,20k-30k,93007,16004,17.21
7,Good,10k-20k,159114,25520,16.04
8,Excellent,30k+,10856,1649,15.19
9,Good,5k-10k,86787,11075,12.76


### Higher-Risk Borrower Segments

To identify more specific risk segments, FICO categories were combined with loan amount bands.

Only segments containing at least 1,000 loans were included to avoid drawing conclusions from very small groups.

#### Key Findings

- Fair-FICO borrowers with loans above 30k had the highest observed default rate at **29.70%**.
- Fair-FICO borrowers with loans between 20k and 30k had a default rate of **27.83%**.
- Fair-FICO borrowers with loans between 10k and 20k had a default rate of **25.02%**.
- Good-FICO borrowers with loans above 30k had a default rate of **19.48%**.
- Excellent-FICO borrowers showed substantially lower default rates across all loan amount bands.

#### Business Interpretation

The analysis suggests that **credit quality and loan size together provide stronger risk segmentation than either variable alone**.

Fair-FICO borrowers receiving larger loans represent a relatively higher-risk segment in this dataset.

However, these results show association rather than causation. Further statistical analysis and machine-learning modeling will be required to determine whether these variables independently contribute to default risk.

### Business Question 12 — High-Risk Segment Loan Share

The previous analysis identified borrower segments by combining FICO score category and loan amount band.

To estimate how much of the portfolio belongs to comparatively higher-risk segments, segments will be considered higher-risk when:

- The segment contains at least 1,000 loans.
- The segment's default rate is above the overall portfolio default rate.

This avoids classifying a segment as high-risk solely because it has a sufficiently large sample size.

### Business Question

What proportion of the total loan portfolio belongs to borrower segments with above-average observed default rates?

In [35]:
high_risk_loan_share = pd.read_sql_query(
    """
    WITH segment_analysis AS (
        SELECT
            CASE
                WHEN fico_n < 650 THEN 'Poor'
                WHEN fico_n BETWEEN 650 AND 699 THEN 'Fair'
                WHEN fico_n BETWEEN 700 AND 749 THEN 'Good'
                ELSE 'Excellent'
            END AS fico_category,

            CASE
                WHEN loan_amnt < 5000 THEN '<5k'
                WHEN loan_amnt BETWEEN 5000 AND 9999 THEN '5k-10k'
                WHEN loan_amnt BETWEEN 10000 AND 19999 THEN '10k-20k'
                WHEN loan_amnt BETWEEN 20000 AND 29999 THEN '20k-30k'
                ELSE '30k+'
            END AS loan_amount_band,

            COUNT(*) AS segment_loans,

            SUM(
                CASE
                    WHEN "Default" = 1 THEN 1
                    ELSE 0
                END
            ) AS segment_defaults,

            ROUND(
                SUM(
                    CASE
                        WHEN "Default" = 1 THEN 1
                        ELSE 0
                    END
                ) * 100.0 / COUNT(*),
                2
            ) AS segment_default_rate

        FROM loans

        GROUP BY fico_category, loan_amount_band

        HAVING COUNT(*) >= 1000
    ),

    portfolio_rate AS (
        SELECT
            SUM(
                CASE
                    WHEN "Default" = 1 THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*) AS overall_default_rate
        FROM loans
    ),

    high_risk_segments AS (
        SELECT
            segment_loans
        FROM segment_analysis
        CROSS JOIN portfolio_rate
        WHERE segment_default_rate > overall_default_rate
    )

    SELECT
        SUM(segment_loans) AS high_risk_segment_loans,

        (SELECT COUNT(*) FROM loans) AS total_loans,

        ROUND(
            SUM(segment_loans) * 100.0 /
            (SELECT COUNT(*) FROM loans),
            2
        ) AS high_risk_loan_share_pct

    FROM high_risk_segments;
    """,
    conn
)

high_risk_loan_share

,high_risk_segment_loans,total_loans,high_risk_loan_share_pct
0,522692,1347681,38.78


###  Interpretation

The analysis identifies loan segments whose observed default rate is higher than the overall portfolio default rate.

These higher-risk segments contain 522,692 loans, representing 38.78% of the total loan portfolio.

This indicates that a substantial portion of the portfolio falls into segments with above-average observed default rates.

This result is useful for portfolio-level risk monitoring because it shows the scale of loans exposed to comparatively higher observed default risk.

However, these segments should not be treated as automatically unacceptable or causal risk groups. The analysis is based on historical observed default rates and should be combined with statistical analysis and machine-learning results in later phases.

### Business Insight

A significant share of the portfolio belongs to segments with above-average observed default rates, suggesting that borrower credit quality and loan size together can help identify areas requiring closer risk monitoring.